## Data Preparation

(download data from torrens: https://velog.io/@jasonlee1995/Linux-Server-Download-ImageNet-1K)

(good resources to learn from: https://github.com/Jasonlee1995/awesome-ai-courses)

In [1]:
from __future__ import annotations

import os
from pathlib import Path

import torch
from torch.utils.data import DataLoader, Subset
from torchvision import datasets
from torchvision.datasets import ImageFolder
from torchvision.transforms import v2
from collections import defaultdict

In [2]:
# Seed every RNG so runs are comparable: CIFAR-10 seed variance (~±0.3-0.7%)
# is the same magnitude as many single-change effects.
import random
import numpy as np

SEED = 42

def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()

### Load to datasets

In [3]:
IMAGENET_DIR = './dataset/Imagenet_1k_extract'
TRAIN_DIR = os.path.join(IMAGENET_DIR, 'train')
VAL_DIR   = os.path.join(IMAGENET_DIR, 'val')

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

In [4]:
BATCH_SIZE = 128
IMAGE_SIZE = 224

train_transform = v2.Compose([
    v2.ToImage(),

    # Randomly crop and resize an ImageNet image to 224 × 224.
    v2.RandomResizedCrop(
        size=(IMAGE_SIZE, IMAGE_SIZE),
        scale=(0.08, 1.0),
        ratio=(3 / 4, 4 / 3),
        antialias=True,
    ),

    v2.RandomHorizontalFlip(p=0.5),

    # Convert uint8 [0, 255] to float32 [0, 1].
    v2.ToDtype(torch.float32, scale=True),

    v2.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD,
    ),

    # Value 0 corresponds approximately to the normalized mean.
    # v2.RandomErasing(
    #     p=0.25,
    #     scale=(0.02, 0.20),
    #     ratio=(0.3, 3.3),
    #     value=0,
    # ),
])

val_transform = v2.Compose([
    v2.ToImage(),
    v2.Resize(256, antialias=True),
    v2.CenterCrop((IMAGE_SIZE, IMAGE_SIZE)),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD,
    ),
])

In [5]:
class NumericImageFolder(ImageFolder):
    """ImageFolder that interprets numeric folder names numerically."""

    def find_classes(
        self,
        directory: str | Path,
    ) -> tuple[list[str], dict[str, int]]:
        classes = [
            entry.name
            for entry in os.scandir(directory)
            if entry.is_dir()
        ]

        if not classes:
            raise FileNotFoundError(
                f"No class directories found in {directory}"
            )

        try:
            classes = sorted(classes, key=int)
        except ValueError as exc:
            raise ValueError(
                "NumericImageFolder requires class directories named "
                "'0', '1', ..., '999'."
            ) from exc

        class_to_idx = {
            class_name: int(class_name)
            for class_name in classes
        }

        expected_classes = list(range(len(classes)))
        actual_classes = [int(name) for name in classes]

        if actual_classes != expected_classes:
            raise ValueError(
                "Class directories must form a continuous numeric range. "
                f"Expected 0..{len(classes) - 1}."
            )

        return classes, class_to_idx

In [6]:
train_dataset = NumericImageFolder(
    root=TRAIN_DIR,
    transform=train_transform,
)

val_dataset = NumericImageFolder(
    root=VAL_DIR,
    transform=val_transform,
)

eval_train_dataset = NumericImageFolder(
    root=TRAIN_DIR,
    transform=val_transform,
)

In [7]:
def make_balanced_subset(
    dataset: NumericImageFolder,
    samples_per_class: int = 10,
    seed: int = 42,
) -> Subset:
    """
    Create a reproducible subset containing the same number of
    samples from every class.

    For ImageNet-1K:
        10 samples/class × 1,000 classes = 10,000 samples.
    """
    indices_by_class: dict[int, list[int]] = defaultdict(list)

    for sample_index, target in enumerate(dataset.targets):
        indices_by_class[target].append(sample_index)

    generator = torch.Generator().manual_seed(seed)
    selected_indices: list[int] = []

    for class_index in range(len(dataset.classes)):
        class_indices = indices_by_class[class_index]

        if len(class_indices) < samples_per_class:
            raise ValueError(
                f"Class {class_index} only contains "
                f"{len(class_indices)} samples, but "
                f"{samples_per_class} were requested."
            )

        permutation = torch.randperm(
            len(class_indices),
            generator=generator,
        )

        selected_indices.extend(
            class_indices[index]
            for index in permutation[:samples_per_class].tolist()
        )

    return Subset(
        dataset=dataset,
        indices=selected_indices,
    )


eval_train_subset = make_balanced_subset(
    dataset=eval_train_dataset,
    samples_per_class=10,
    seed=SEED,
)


In [8]:
if train_dataset.class_to_idx != val_dataset.class_to_idx:
    raise ValueError(
        "Training and validation class mappings do not match."
    )

if train_dataset.class_to_idx != eval_train_dataset.class_to_idx:
    raise ValueError(
        "Training and evaluation class mappings do not match."
    )

print(f"Training images:          {len(train_dataset):,}")
print(f"Validation images:        {len(val_dataset):,}")
print(f"Evaluation subset images: {len(eval_train_subset):,}")
print(f"Number of classes:        {len(train_dataset.classes):,}")

Training images:          1,281,167
Validation images:        50,000
Evaluation subset images: 10,000
Number of classes:        1,000


### Create data loaders

In [9]:
NUM_WORKERS = min(8, os.cpu_count() or 1)
PIN_MEMORY = torch.cuda.is_available()

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=NUM_WORKERS > 0,
    prefetch_factor=2 if NUM_WORKERS > 0 else None,
    drop_last=True,
)

val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=NUM_WORKERS > 0,
    prefetch_factor=2 if NUM_WORKERS > 0 else None,
    drop_last=False,
)

eval_train_loader = DataLoader(
    dataset=eval_train_subset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    drop_last=False,
)

## Models

In [10]:
import torch
import torch.nn.functional as F
from torch import nn

import math
import os
import time
from datetime import datetime

import numpy as np
import h5py
import matplotlib.pyplot as plt
from matplotlib.pyplot import imread
import scipy
from PIL import Image
import pandas as pd

from typing import Sequence

### Resnet

In [11]:
class BasicResidualBlock(nn.Module):
    """
        Canonical basic block (for resnet18, resnet34), He et al. 2015:

        input > 3×3 Conv (stride) > BN > ReLU > 3×3 Conv (1) > BN > Add(identity) > ReLU

        Convs carry no bias: the BatchNorm right after each conv has its own shift,
        so a conv bias would be a dead parameter.

        *In this case, the cifar10 dataset is not complicated enough for the bottleneck block to make any differences. > use basic block for now.*
    """

    expansion = 1

    def __init__(self, 
                 input_dim: int, # number of channels
                 planes: int, # internal width per stage in net
                 stride: int = 1, # initial stride    
                 downsample: nn.Module = None, # block of shortcut
        ):
        super(BasicResidualBlock, self).__init__()

        output_channels = planes * self.expansion

        self.residual = nn.Sequential(
            nn.Conv2d(input_dim, planes, kernel_size=3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(planes),
            nn.ReLU(inplace=True),

            nn.Conv2d(planes, output_channels, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(output_channels),
        )

        if downsample:
            self.downsample = downsample
        else:
            self.downsample = nn.Identity()
    
    def forward(self, x):
        identity = self.downsample(x)
        residual = self.residual(x)
        # Post-add activation: out = relu(F(x) + x). The nonlinearity must sit
        # AFTER the addition, otherwise blocks can only ever add non-negative values.
        return F.relu(identity + residual)

class resnet(nn.Module):
    def __init__(self,
        Block,
        layers: Sequence[int],
        num_classes: int = 10,
        input_channels: int = 3):

        super(resnet, self).__init__()

        self.in_channels = 64
        # CIFAR stem (He et al. sec 4.2): 3×3 stride-1 conv, no maxpool.
        # The ImageNet stem (7×7 s2 + maxpool) would shrink 32×32 inputs to 8×8
        # before the first residual block, leaving the deep stages with 2×2/1×1 maps.
        self.conv1 = nn.Sequential(
            nn.Conv2d(input_channels, 64, kernel_size=7, stride=2, padding=3, bias=False,),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
        )

        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1,)

        # Layers — stage resolutions on CIFAR: 56 > 28 > 14 > 7
        self.big_layers = nn.Sequential(
            self._make_layer(Block=Block, planes=64, number_of_blocks=layers[0], stride=1),
            self._make_layer(Block=Block, planes=128, number_of_blocks=layers[1], stride=2),
            self._make_layer(Block=Block, planes=256, number_of_blocks=layers[2], stride=2),
            self._make_layer(Block=Block, planes=512, number_of_blocks=layers[3], stride=2),   
        )

        self.avgpool = nn.AdaptiveAvgPool2d((1,1))

        # Single linear head: the 512-d pooled vector is already a summary,
        # a deep MLP here is pure memorization capacity.
        self.classification_head = nn.Linear(512 * Block.expansion, num_classes)

        # Kaiming fan-out init for ReLU conv stacks (PyTorch default is fan-in
        # with a=sqrt(5), which under-scales). BN starts at gamma=1, beta=0.
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

        # Zero-init the LAST BN gamma of each block so every block starts as an
        # identity mapping (Goyal et al. 2017) — stabilises early high-LR training.
        for m in self.modules():
            if isinstance(m, BasicResidualBlock):
                nn.init.zeros_(m.residual[4].weight)

    def _make_layer(self,
        Block: BasicResidualBlock,
        planes: int,
        number_of_blocks: int,
        stride: int = 1):

        output_channels = planes * Block.expansion

        layers = []
        downsample = None

        if stride != 1 or self.in_channels != output_channels:
            # Shortcut projection uses the SAME normalization as the residual
            # branch (BN), so both paths stay in one statistics regime.
            downsample = nn.Sequential(
                nn.Conv2d(self.in_channels, output_channels, kernel_size=1, stride=stride, padding=0, bias=False),
                nn.BatchNorm2d(output_channels),
            )

        layers.append(Block(input_dim=self.in_channels, 
                         planes=planes, 
                         downsample=downsample,
                         stride=stride))

        self.in_channels = output_channels

        for i in range(1, number_of_blocks):
            layers.append(Block(self.in_channels, planes=planes))

        return nn.Sequential(*layers)
    
    def forward(self, x):
        x = self.conv1(x)
        x = self.maxpool(x)
        x = self.big_layers(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)  # Flatten the tensor
        return self.classification_head(x)


## Training Setup

### Resnet

In [12]:
input_dim = 3
output_dim = 1000

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device, torch.cuda.get_device_name(0) if device.type == "cuda" else "")

torch.backends.cudnn.benchmark = True  # fixed input size -> let cudnn pick the fastest kernels

set_seed()  # seed right before weight init so the run is reproducible
net = resnet(Block=BasicResidualBlock, 
             layers=[3, 4, 6, 3],  
             num_classes=output_dim,
             input_channels=input_dim).to(device)
net = net.to(memory_format=torch.channels_last)
# net = torch.compile(net, fullgraph=True)

print(f"{sum(p.numel() for p in net.parameters()) / 1e6:.2f}M parameters")

cuda NVIDIA GeForce RTX 5080
21.80M parameters


In [13]:
net

resnet(
  (conv1): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU(inplace=True)
  )
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (big_layers): Sequential(
    (0): Sequential(
      (0): BasicResidualBlock(
        (residual): Sequential(
          (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
          (2): ReLU(inplace=True)
          (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        )
        (downsample): Identity()
      )
      (1): BasicResidualBlock(
        (residual): Sequential(
     

#### Test (data loader + model before training)

In [14]:
images, targets = next(iter(train_loader))

print("Image batch:", images.shape)
print("Target batch:", targets.shape)
print("Image dtype:", images.dtype)
print("Target dtype:", targets.dtype)
print("Target range:", targets.min().item(), targets.max().item())

Image batch: torch.Size([128, 3, 224, 224])
Target batch: torch.Size([128])
Image dtype: torch.float32
Target dtype: torch.int64
Target range: 42 998


In [15]:
images = images.to(
    device,
    non_blocking=True,
)

if device.type == "cuda":
    images = images.contiguous(
        memory_format=torch.channels_last
    )

targets = targets.to(
    device,
    non_blocking=True,
)

net.eval()

with torch.inference_mode():
    logits = net(images)

print("Output:", logits.shape)

assert logits.shape == (
    images.size(0),
    output_dim,
)

Output: torch.Size([128, 1000])


## Run

In [16]:
epochs = 100

# Label smoothing caps logit over-confidence; the printed loss will floor
# near ~0.5 instead of 0 — that is expected, not a bug.
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# SGD + momentum is the canonical CIFAR ResNet recipe (the previous AdamW runs
# showed the classic Adam generalization deficit: train 92.5% / test 82.5%).
# Weight decay goes on conv/linear weights ONLY: decaying BN affine params and
# biases (everything with ndim == 1) fights the normalization instead of regularizing.
decay, no_decay = [], []
for name, p in net.named_parameters():
    (no_decay if p.ndim == 1 else decay).append(p)

# optimizer = torch.optim.SGD(
#     [{"params": decay, "weight_decay": 5e-4},
#      {"params": no_decay, "weight_decay": 0.0}],
#     lr=0.1,  # calibrated to BATCH_SIZE = 128
#     momentum=0.9,
#     nesterov=True,
# )

optimizer = torch.optim.AdamW(net.parameters(), lr=1e-3, weight_decay=1e-2)

# 5-epoch linear warmup (0.01 -> 0.1) guards against early divergence at lr=0.1,
# then cosine decay to 0. Stepped once per epoch in run().
warmup = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, total_iters=5)
cosine = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs - 5, eta_min=0.0)
scheduler = torch.optim.lr_scheduler.SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[5])

In [17]:
def check_accuracy(loader, model):
    num_correct = 0
    num_samples = 0

    model.eval()

    with torch.inference_mode():
        for x, y in loader:
            x = x.to(device, non_blocking=True, memory_format=torch.channels_last)
            y = y.to(device, non_blocking=True)

            with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
                scores = model(x)
            _, predictions = scores.max(1)

            num_correct += (predictions == y).sum().item()
            num_samples += predictions.size(0)

    return num_correct / num_samples

In [18]:
def run(model, optimizer, scheduler, criterion, epochs, writer, run_name):
    # AMP: forward in reduced precision, gradients rescaled to avoid underflow.
    # enabled= flags keep the whole loop runnable on CPU too.
    scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))

    os.makedirs("checkpoints", exist_ok=True)
    best_acc = 0.0
    step = 0

    try:
        for epoch in range(epochs):
            t0 = time.perf_counter()
            running_loss = 0.0
            model.train()

            for data, targets in train_loader:
                data = data.to(device, non_blocking=True, memory_format=torch.channels_last)
                targets = targets.to(device, non_blocking=True)

                # Forward propagation
                with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
                    scores = model(data)
                    loss = criterion(scores, targets)

                # Backward propagation
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

                running_loss += loss.item()

                if step % 50 == 0:
                    writer.add_scalar("Loss/train_batch", loss.item(), step)
                step += 1

            avg_epoch_loss = running_loss / len(train_loader)

            # Train accuracy on the clean fixed subset, test on the full test set.
            train_acc = check_accuracy(eval_train_loader, model)
            test_acc = check_accuracy(val_loader, model)

            # Step the scheduler once per epoch
            scheduler.step()

            # Log the learning rate used for the next epoch
            current_lr = optimizer.param_groups[0]["lr"]

            epoch_time = time.perf_counter() - t0
            writer.add_scalar("Loss/train_epoch", avg_epoch_loss, epoch)
            writer.add_scalar("Accuracy/train", train_acc, epoch)
            writer.add_scalar("Accuracy/test", test_acc, epoch)
            writer.add_scalar("Accuracy/gap", train_acc - test_acc, epoch)
            writer.add_scalar("LR", current_lr, epoch)
            writer.add_scalar("Time/epoch_sec", epoch_time, epoch)

            # Keep the best weights — cosine-to-zero means the last epoch is
            # usually the best, but this run is insurance against surprises.
            if test_acc > best_acc:
                best_acc = test_acc
                torch.save({
                    "epoch": epoch,
                    "test_acc": test_acc,
                    "model_state": model.state_dict(),
                    "optimizer_state": optimizer.state_dict(),
                    "scheduler_state": scheduler.state_dict(),
                }, f"checkpoints/{run_name}_best.pt")

            print(
                f"Epoch [{epoch + 1}/{epochs}], "
                f"Loss: {avg_epoch_loss:.4f}, "
                f"Train Acc: {train_acc:.4f}, "
                f"Test Acc: {test_acc:.4f}, "
                f"LR: {current_lr:.6f}, "
                f"{epoch_time:.1f}s"
            )
    finally:
        torch.save(model.state_dict(), f"checkpoints/{run_name}_final.pt")
        writer.close()

    print(f"Best test acc: {best_acc:.4f} (checkpoints/{run_name}_best.pt)")

### Run Resnet

In [ ]:
import json
from torch.utils.tensorboard import SummaryWriter

# One directory per run so TensorBoard curves never overlap between runs.
run_name = f"{datetime.now():%Y%m%d-%H%M%S}_resnet34"
writer = SummaryWriter(f"runs/imagenet/{run_name}")
writer.add_text("config", json.dumps({
    "arch": "resnet34-cifar", "layers": [3, 4, 6, 3],
    "optimizer": "AdamW", "lr": 1e-3, "weight_decay": 1e-2, 
    "scheduler": "warmup5+cosine", "epochs": epochs, "batch_size": BATCH_SIZE,
    "label_smoothing": 0.1, "augmentation": "crop-reflect+hflip+erasing", "seed": SEED,
}))

run(net, optimizer=optimizer, scheduler=scheduler, criterion=criterion,
    epochs=epochs, writer=writer, run_name=run_name)

## Load from best pt

In [ ]:
# best_acc = 0.0
# start_epoch = 0
# ckpt = torch.load(args.resume, map_location=device, weights_only=False)
# if isinstance(ckpt, dict) and "model_state" in ckpt:
#     net.load_state_dict(ckpt["model_state"])
#     optimizer.load_state_dict(ckpt["optimizer_state"])
#     best_acc = float(ckpt.get("test_acc", 0.0))
#     if not args.fresh_schedule:
#         if "scheduler_state" in ckpt:
#             scheduler.load_state_dict(ckpt["scheduler_state"])
#         start_epoch = int(ckpt.get("epoch", 0)) + 1  # continue after the saved epoch
#     print(f"Resumed {args.resume}: epoch {ckpt.get('epoch')} "
#             f"test_acc {best_acc:.4f} -> start_epoch {start_epoch} "
#             f"({'fresh schedule' if args.fresh_schedule else 'resumed schedule'})", flush=True)
# else:
#     net.load_state_dict(ckpt)
#     print(f"Loaded raw state_dict {args.resume} (weights only)", flush=True)

# run_name = f"{datetime.now():%Y%m%d-%H%M%S}_resnet34_resume"
# writer = SummaryWriter(f"runs/imagenet/{run_name}")
# writer.add_text("config", json.dumps({
#     "arch": "resnet34-imagenet", "layers": [3, 4, 6, 3],
#     "optimizer": "AdamW", "lr": 1e-3, "weight_decay": 1e-2,
#     "scheduler": "warmup5+cosine", "epochs": args.epochs, "batch_size": BATCH_SIZE,
#     "label_smoothing": 0.1, "seed": SEED, "resume_from": args.resume,
#     "start_epoch": start_epoch, "fresh_schedule": args.fresh_schedule,
# }))

# run(net, optimizer, scheduler, criterion, args.epochs, writer, run_name,
#     loaders, device, start_epoch=start_epoch, best_acc=best_acc)


RuntimeError: DataLoader worker (pid(s) 178043, 178044, 178045, 178046, 178047, 178048, 178049, 178050) exited unexpectedly